In [ ]:
# load the correlation dataset (median multiple and gdp)

GDP_MM_Correlation_df = pd.read_csv('data/GDP_MM_processed.csv')

GDP_MM_Correlation_df.head()

In [ ]:
# import Federal Funds Rate data
# Considerations: FFR is used to act upon the economy
# "Like the federal discount rate, the federal funds rate is used to 
# control the supply of available funds and hence, inflation and other interest rates."
# Source: https://www.bankrate.com/rates/interest-rates/federal-funds-rate/
# Consider the current observation date taking between 6-18 months to 
# impact the economy so may want to create a lag where this data
# is set backwords 6, 12, and 18 months
# This data is given at 1 month intervals and so will need to be
# changed to quarterly intervals and imputed

fed_funds_df = pd.read_csv('/Users/douglas/VS_Code/FEDFUNDS.csv')
income_df.info

# create a copy of raw data
fed_funds_df_copy = fed_funds_df

# Change object to datetime
fed_funds_df_copy['observation_date'] = pd.to_datetime(fed_funds_df_copy['observation_date'])

# rename column for better readability
fed_funds_df_copy = fed_funds_df_copy.rename(columns = {'FEDFUNDS': 'fed_funds_rate'})

# Filter the data to match the existing time interval
fed_funds_df_copy = fed_funds_df_copy[(fed_funds_df_copy['observation_date'] >= '1962-07-01') & (fed_funds_df_copy['observation_date'] <= '2023-01-01')]

# create quarterly intervals instead of yearly
rates_quarterly = fed_funds_df_copy.groupby(pd.Grouper(key='observation_date', freq="QS"))['fed_funds_rate'].mean()
fed_funds_df_copy = rates_quarterly.to_frame().reset_index()

# fed_funds_df_copy.head()

# Create lag dataframes
fed_funds_6m = fed_funds_df_copy.copy()
fed_funds_6m['observation_date'] = fed_funds_6m['observation_date'] - pd.DateOffset(months=6)
fed_funds_6m = fed_funds_6m.rename(columns={'fed_funds_rate': 'fed_funds_rate_6m'})

fed_funds_12m = fed_funds_df_copy.copy()
fed_funds_12m['observation_date'] = fed_funds_12m['observation_date'] - pd.DateOffset(months=12)
fed_funds_12m = fed_funds_12m.rename(columns={'fed_funds_rate': 'fed_funds_rate_12m'})

fed_funds_18m = fed_funds_df_copy.copy()
fed_funds_18m['observation_date'] = fed_funds_18m['observation_date'] - pd.DateOffset(months=18)
fed_funds_18m = fed_funds_18m.rename(columns={'fed_funds_rate': 'fed_funds_rate_18m'})

# Merge X2 data to use in multiple linear regression with subset selection.




In [ ]:
# Troubleshoot the input variable to match number of records and proper dates

# fed_funds_18m.head     # 243 rows which does not match the 241 needed
# results of head function show start dates as follows
# bound method NDFrame.head of     observation_date  fed_funds_rate_18m
# 0         1961-01-01            2.846667             # incorrect start date - must trim to  1961-07-01
# 1          1961-04-01            2.923333
# 2         1961-07-01            2.966667


# Calculate the 18 month date prior to the first date in X1 data (GDP)
GDP_df_copy.head() # start date of 1963-01-01 
# check 18 months prior 
print(GDP_df_copy['observation_date'] - pd.DateOffset(months=18)) # first date should be 1961-07-01
# by trimming these first two dates off we will have a matching number of records
# fed_funds_18m['observation_date'] - pd.DateOffset(months=18)

fed_funds_18m = fed_funds_18m[(fed_funds_18m['observation_date'] >= '1961-07-01')]
# fed_funds_18m.head()
# fed_funds_18.count


In [ ]:
# repeat above process with 6 and 12 month lags
# fed_funds_12m.head() # start date: 1961-07-01	

# calculate proper start dates
# print(GDP_df_copy['observation_date'] - pd.DateOffset(months=6)) # first date should be 1962-07-01
# print(GDP_df_copy['observation_date'] - pd.DateOffset(months=12)) # first date should be 1962-01-01

fed_funds_6m = fed_funds_6m[(fed_funds_6m['observation_date'] >= '1962-07-01')]
# fed_funds_6m.head() # verified correct start date
# fed_funds_6m.count() # verified correct count

fed_funds_12m = fed_funds_12m[(fed_funds_12m['observation_date'] >= '1962-01-01')]
# fed_funds_12m.head() # verified correct start date
# fed_funds_12m.count() # verified correct count

In [ ]:
fed_funds_6m = fed_funds_6m[(fed_funds_6m['observation_date'] >= '1962-07-01')]
# fed_funds_6m.head() # verified correct start date
fed_funds_18m.count() # verified correct count

In [ ]:
# Create X_subset_fed_funds_lag as a DataFrame
X_subset_fed_funds_lag = pd.DataFrame({
    'fed_funds_rate_6m': fed_funds_6m['fed_funds_rate_6m'],
    'fed_funds_rate_12m': fed_funds_12m['fed_funds_rate_12m'],
    'fed_funds_rate_18m': fed_funds_18m['fed_funds_rate_18m']
})

# Create Y_fed_funds
Y_fed_funds = df_merged["Median_Multiple"]

# Check if lengths match
if len(Y_fed_funds) != len(X_subset_fed_funds_lag):
    raise ValueError("Length of Y_fed_funds does not match length of X_subset_fed_funds_lag.")

# Reset indices for X and Y
X_subset_fed_funds_lag = X_subset_fed_funds_lag.reset_index(drop=True)
Y_fed_funds = Y_fed_funds.reset_index(drop=True)

In [ ]:
import statsmodels.api as sm

# List of predictors
predictors = ['fed_funds_rate_6m', 'fed_funds_rate_12m', 'fed_funds_rate_18m']

# Dictionary to store results
results = {}

# Fit models for each predictor
for predictor in predictors:
    X = sm.add_constant(X_subset_fed_funds_lag[predictor])  # Add intercept
    model = sm.OLS(Y_fed_funds, X).fit()
    results[predictor] = {
        'R-squared': model.rsquared,
        'Adjusted R-squared': model.rsquared_adj,
        'AIC': model.aic
    }

# Print results
for predictor, metrics in results.items():
    print(f"Predictor: {predictor}")
    print(f"  R-squared: {metrics['R-squared']:.4f}")
    print(f"  Adjusted R-squared: {metrics['Adjusted R-squared']:.4f}")
    print(f"  AIC: {metrics['AIC']:.4f}")
    print()

# Find the best predictor based on adjusted R-squared
best_predictor = max(results, key=lambda x: results[x]['Adjusted R-squared'])
print(f"Best predictor: {best_predictor}")

In [ ]:
print(X_subset_fed_funds_lag.corr())

In [ ]:
# import Consumer Price Index for All Urban Consumers: All Items in U.S. City Average
# This data indirectly includes information on the cost of housing.
# There may be issues with using it as a predictor of MM due to this

X3_df = pd.read_csv('/Users/douglas/VS_Code/CPIAUCSL.csv')


# create a copy of raw data
X3_df_copy = X3_df

X3_df_copy.head()
X3_df_copy.describe()

# Change object to datetime
X3_df_copy['observation_date'] = pd.to_datetime(X3_df_copy['observation_date'])

# rename column for better readability
X3_df_copy = X3_df_copy.rename(columns = {'CPIAUCSL': 'consumer_price_index'})

# create quarterly intervals instead of yearly
cpi_quarterly = X3_df_copy.groupby(pd.Grouper(key='observation_date', freq="QS"))['consumer_price_index'].mean()
X3_df_copy = cpi_quarterly.to_frame().reset_index()

# Filter the data to match the existing time interval
X3_df_copy = X3_df_copy[(X3_df_copy['observation_date'] >= '1963-01-01') & (X3_df_copy['observation_date'] <= '2023-01-01')]
# X3_df_copy.head()



In [ ]:
# import US Unemployment Rate data
# This data is monthly and needs to be changed to Quarterly and imputed.


X4_df = pd.read_csv('/Users/douglas/VS_Code/UNRATE.csv')
income_df.info

# create a copy of raw data
X4_df_copy = X4_df

# X4_df_copy.head()
# X4_df_copy.describe()

# Change object to datetime
X4_df_copy['observation_date'] = pd.to_datetime(X4_df_copy['observation_date'])

# rename column for better readability
X4_df_copy = X4_df_copy.rename(columns = {'UNRATE': 'unemployment_rate'})

# create quarterly intervals instead of yearly
unemployement_quarterly = X4_df_copy.groupby(pd.Grouper(key='observation_date', freq="QS"))['unemployment_rate'].mean()
X4_df_copy = unemployement_quarterly.to_frame().reset_index()

# Filter the data to match the existing time interval
X4_df_copy = X4_df_copy[(X4_df_copy['observation_date'] >= '1963-01-01') & (X4_df_copy['observation_date'] <= '2023-01-01')]
X4_df_copy.count()


In [ ]:
import statsmodels.api as sm

# Select the predictor
predictor = X4_df_copy

# Prepare the data
X = sm.add_constant(X4_df_copy['unemployment_rate'])  # Add intercept
# Create Y
Y = df_merged["Median_Multiple"]

# Reset indices for X and Y
X = X.reset_index(drop=True)
Y = Y.reset_index(drop=True)

# Fit the model
model = sm.OLS(Y, X).fit()

# Print model summary
print(model.summary())

# Extract key metrics
r_squared = model.rsquared
adjusted_r_squared = model.rsquared_adj
aic = model.aic

print(f"R-squared: {r_squared:.4f}")
print(f"Adjusted R-squared: {adjusted_r_squared:.4f}")
print(f"AIC: {aic:.4f}")

## Older Markdown (need to be updated with results from new methodology and new observations)

Discussion regarding unemployement rate:

Unemployement rate offers no predictive insight on housing affordability. 
This may be a relevent consideration regarding the effectiveness of interest rate hikes to contol inflation in the sphere of home affordability. Interest rate hikes can lead to increased unemployment rates which contribute to cooling demand. 


Conclusion:

The analysis reveals a surprisingly strong inverse relationship between real GDP growth and home affordability, as demonstrated through both linear regression and decision tree classification using the Median Multiple metric. The decision tree model proved particularly interesting due to its classification capabilities, with its exceptional accuracy potentially attributed to its nonparametric nature. While the linear relationship demonstrates very strong statistical significance, there is evidence that events such as the Great Financial Crisis, changes to supply chain dynamics, and global alliance shifts starting in the COVID-19 pandemic suggest an optimal model may require additional predictor variables and reduced bias.

This counterintuitive relationship merits further investigation and expert consultation. Future research should explore whether this pattern generalizes across different time periods and geographical regions, as well as whether certain sectors of GDP growth provide more reliable models under different conditions. Although housing remained broadly affordable for most of the population until approximately 2010, the early warning signs of this inverse relationship deserved greater attention.

These findings challenge GDP's adequacy as the primary indicator of economic health and suggest the need for more comprehensive models. The deteriorating housing affordability despite GDP growth illuminates some root causes of current political instability within liberal democracies and suggests a need for reconsideration of fundamental economic assumptions. Without thoughtful constraints, economic freedom may paradoxically undermine the stability of the liberal world order.

At an intuitive level, we expect increased productive output to yield better outcomes—just as maintaining a car or home typically improves its condition. When our primary measure of economic production fails to deliver widely beneficial results, we must critically evaluate its merits. The steadily worsening median multiple over time represents a troubling trend with evident connections to today's major domestic and geopolitical shifts.

Sources:

Nasiha Salwati, D. W., Tyler Powell, D. W., Michael Ng, D. W., Elaine Kamarck, W. A. G., & Alan J. Auerbach, W. G. G. (2024, January 31). How does the consumer price index account for the cost of housing?. Brookings. https://www.brookings.edu/articles/how-does-the-consumer-price-index-account-for-the-cost-of-housing/ 

Callen, T. (2008, December). Back to basics: What is gross domestic product?. imf.org. https://www.imf.org/external/pubs/ft/fandd/2008/12/pdf/basics.pdf 

COX, W. (2024). Demographia International Housing Affordability – 2024 edition released. Demographia International Housing Affordability – 2024 Edition Released | Newgeography.com. https://www.newgeography.com/content/008198-demographia-international-housing-affordability-2024-edition-released 

Grossman, D. (2024, September 10). Where you live may dictate whether you’re middle-class. Scripps News. https://www.scrippsnews.com/life/money/mastering-your-money/where-you-live-may-dictate-whether-youre-middle-class 

Galston, W. A. (2020, July). The enduring vulnerability of Liberal Democracy. Journal of Democracy. https://www.journalofdemocracy.org/articles/the-enduring-vulnerability-of-liberal-democracy/ 

Stobierski, T. (2021, June 8). What is GDP & why is it important?. Business Insights Blog. https://online.hbs.edu/blog/post/why-is-gdp-important 

Galston, W. A. (2020, July). The enduring vulnerability of Liberal Democracy. Journal of Democracy. https://www.journalofdemocracy.org/articles/the-enduring-vulnerability-of-liberal-democracy/ 


Datasets:

Y =
median multiple:
MM equation = medium_homeprice / median_income
= [
Median Home Price Data:
source: https://fred.stlouisfed.org/series/MSPUS
Quarterly since 1963 - 2024-10-01
FileName: 1963-01-01_Med_Home_Price
/
Real Median Family Income in the United States data source:
source: https://fred.stlouisfed.org/series/MEFAINUSA672N#
date range...yearly 1953-2023-01-01
FileName: MedianUS_income1953
]

X1 
Data for...
Real Gross Domestic Product 
Source: https://fred.stlouisfed.org/series/GDPC1#
Saved as...GDPC1-2
date range...1947-01-01 - 2024-10-01
Units: Billions of Chained 2017 Dollars | Quarterly

X2 (exploratory)
Data for... 
Federal Funds Rate
Source: https://fred.stlouisfed.org/series/FEDFUNDS
date range: Monthly 1954-07-01 - 2025-02-01
Name: FEDFUNDS
Units: Percent | Monthly
more info: https://www.bankrate.com/rates/interest-rates/federal-funds-rate/

X3 (exploratory)
Data for...
Consumer Price Index for All Urban Consumers: All Items in U.S. City Average (CPIAUCSL)
Source: https://fred.stlouisfed.org/series/CPIAUCSL
date range: monthly 1947-01-01 - 2025
file name: CPIAUCSL

X4 (exploratory)
Data for 
US Unemployment Rate
Source: https://fred.stlouisfed.org/series/UNRATE/
date range: 1948-2025
File Name: UNRATE.csv
Units: Percent, Seasonally Adjusted


